## **EDA & Pre-modeling diagnostics**
- **Statistical summary** of daily log-returns for 4 currency pairs: USD/VND, EUR/VND, JPY/VND, and CNY/VND
- **Objectives:** Verify distribution (normality), serial correlation (linear), and ARCH effects (non-linear)
- **Context:** Supplements ADF/KPSS/Bai-Perron tests for high-fidelity model justification


In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# 1. Coordinate Paths: Move to project root if running from notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


In [ ]:
# 2. Load data
df_train = pd.read_csv("data/processed/train.csv", parse_dates=["Date"])
df_train.set_index("Date", inplace=True)

# 3. Target columns
ret_cols = ["USDVND_RET", "EURVND_RET", "JPYVND_RET", "CNYVND_RET"]
pairs_labels = ["USD/VND", "EUR/VND", "JPY/VND", "CNY/VND"]

results = {}

# 4. Diagnostics Loop
for col, label in zip(ret_cols, pairs_labels):
    series = df_train[col].dropna()

    # Jarque-Bera (Normality)
    jb_stat, jb_p = stats.jarque_bera(series)

    # Ljung-Box (Autocorrelation at Lag 10)
    lb_p = acorr_ljungbox(series, lags=[10])["lb_pvalue"].iloc[0]

    # ARCH-LM (Volatility Clustering at Lag 10)
    _, arch_p, _, _ = het_arch(series, nlags=10)

    results[label] = {
        "Count": series.count(),
        "Mean": series.mean(),
        "Std. Dev.": series.std(),
        "Min": series.min(),
        "Median": series.median(),
        "Max": series.max(),
        "IQR": series.quantile(0.75) - series.quantile(0.25),
        "Skewness": series.skew(),
        "Kurtosis (Excess)": series.kurtosis(),
        "Jarque-Bera (p-val)": jb_p,
        "Ljung-Box (p-val)": lb_p,
        "ARCH-LM (p-val)": arch_p,
    }


In [3]:
# 5. Output Table
pd.options.display.float_format = "{:.4f}".format
diag_table = pd.DataFrame(results)

print("Table 1: Descriptive Stats & Diagnostic Tests")
display(diag_table)


Table 1: Descriptive Stats & Diagnostic Tests


,USD/VND,EUR/VND,JPY/VND,CNY/VND
Count,4966.0000,4966.0000,4966.0000,4966.0000
Mean,0.0054,-0.0001,-0.0032,0.0045
Std. Dev.,0.5517,0.7562,0.7487,0.6152
Min,-8.9352,-9.9874,-9.5084,-9.8573
Median,0.0000,0.0000,0.0000,0.0000
Max,8.9590,8.0508,9.2718,9.2661
IQR,0.0447,0.4421,0.4186,0.1675
Skewness,0.4035,-0.0125,0.0879,0.1527
Kurtosis (Excess),37.8317,13.4157,14.2991,48.6137
Jarque-Bera (p-val),0.0000,0.0000,0.0000,0.0000


- **Non-Normality (JB Test):** Null hypothesis rejected ($p < 0.01$). Excess Kurtosis ($\approx 37.8$) confirms a "heavy-tailed" leptokurtic distribution. This justifies non-parametric models (SVR/MLP) which absorb non-Gaussian error terms.
- **Serial Correlation (LB Test):** Significant linear dependency detected ($p < 0.01$). Log-returns show clear autocorrelation, validating the Autoregressive (ARIMA) and Vector (VAR) stage.
- **Volatility Clustering (ARCH-LM):** Strong non-linear dependency ($p < 0.01$). Predictable variance shifts (heteroscedasticity) justify the Hybrid model’s secondary stage.
- **p-value = 0.0000:** A numerical result of the high test statistics; the probability under a null hypothesis is beyond float precision ($< 10^{-16}$).

### **Distribution Analysis (USD/VND)**
- Visually inspect **fat tails** and **non-normality**
- **Histogram**: Frequency distribution with KDE overlay
- **Q-Q Plot**: Deviations from theoretical Normal distribution


In [4]:
# 1. Data Preparation
val = df_train["USDVND_RET"].dropna()
# Calculate Q-Q points
(osm, osr), (slope, intercept, r) = stats.probplot(val, dist="norm")
line_x = np.array([osm.min(), osm.max()])
line_y = slope * line_x + intercept


In [5]:
# 2. Spacious Horizontal Range (±35% centered at 0)
# This provides generous padding around the bulk of the data
h_range = [-35, 35]
# 3. Setup Figure
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("<b>Return Distribution</b>", "<b>Quantile-Quantile Plot</b>"),
    horizontal_spacing=0.12,
)
# --- Subplot 1: Distribution ---
fig.add_trace(
    go.Histogram(
        x=val,
        nbinsx=500,  # High granularity for the peak
        histnorm="probability density",
        name="Empirical",
        marker_color="#34495e",  # Slightly lighter slate
        opacity=0.85,
    ),
    row=1,
    col=1,
)
# Normal Curve (Reference)
x_range = np.linspace(h_range[0], h_range[1], 1000)
y_norm = stats.norm.pdf(x_range, val.mean(), val.std())
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=y_norm,
        mode="lines",
        name="Normal Approx",
        line=dict(color="#e74c3c", width=2, dash="dot"),
    ),
    row=1,
    col=1,
)
# --- Subplot 2: Q-Q Plot ---
fig.add_trace(
    go.Scatter(
        x=osm,
        y=osr,
        mode="markers",
        name="Quantiles",
        marker=dict(color="#d35400", size=3.5, opacity=0.4),
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        name="Ideal Normal",
        line=dict(color="#2c3e50", width=1.5),
    ),
    row=1,
    col=2,
)
# 4. Global Refinement
fig.update_layout(
    template="plotly_white",
    height=550,
    width=1200,  # Slightly wider
    showlegend=False,
    bargap=0.05,  # Adds spacing between bins for "The Economist" look
    font=dict(family="Arial, sans-serif", size=13, color="#2c3e50"),
    title_text="<b>USD/VND Log-Return Diagnostics</b>",
    title_font_size=20,
    title_x=0.05,
    margin=dict(t=100, b=80, l=80, r=80),
)
# Precision Axis Control
fig.update_xaxes(
    title_text="Daily Return (%)",
    range=h_range,
    row=1,
    col=1,
    showgrid=True,
    gridcolor="#f0f0f0",
    zerolinecolor="#bdc3c7",
)
fig.update_yaxes(title_text="Density", row=1, col=1, showgrid=True, gridcolor="#f0f0f0")
fig.update_xaxes(title_text="Theoretical Quantiles", row=1, col=2, gridcolor="#f0f0f0")
fig.update_yaxes(
    title_text="Ordered Return (%)",
    range=h_range,
    row=1,
    col=2,
    showgrid=True,
    gridcolor="#f0f0f0",
    zerolinecolor="#bdc3c7",
)
fig.show()


- **Leptokurtic Peaking:** The vertical spike at zero confirms that most returns are zero (peg maintenance), making the distribution reach a density $> 11$.
- **Tail Divergence ($Q-Q$ Plot):** The strong "S-curve" reaching $\pm 9\%$ indicates that extreme market jumps are significantly more frequent than predicted by a Normal distribution.
- **Model Justification:** The visual rejection of a Gaussian profile confirms that linear models will suffer from oversized residuals, requiring the help of non-linear Machine Learning.

### **Temporal Dependency Analysis**
- **ACF (Log-Returns):** Visual confirmation of linear autocorrelation (justifies ARIMA/VAR)
- **ACF (Squared Returns):** Visual proof of second-moment dependency (justifies Hybrid/ML logic)


In [6]:
# Calculate ACF values
series = df_train["USDVND_RET"].dropna()
acf_vals = sm.tsa.acf(series, nlags=40)
acf_sq_vals = sm.tsa.acf(series**2, nlags=40)
lags = np.arange(len(acf_vals))


In [7]:
# Setup Subplots
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "<b>ACF: Log-Returns</b>",
        "<b>ACF: Squared Returns (Volatility)</b>",
    ),
    horizontal_spacing=0.12,
)

# Plot ACF
fig.add_trace(
    go.Bar(x=lags, y=acf_vals, marker_color="#2c3e50", name="ACF"), row=1, col=1
)
# Plot Squared ACF
fig.add_trace(
    go.Bar(x=lags, y=acf_sq_vals, marker_color="#d35400", name="Sq ACF"), row=1, col=2
)

# Significance Threshold (approx 95% CI)
ci = 1.96 / np.sqrt(len(series))
for col in [1, 2]:
    fig.add_hline(
        y=ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )
    fig.add_hline(
        y=-ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )

# Styling
fig.update_layout(
    template="plotly_white",
    height=450,
    width=1100,
    showlegend=False,
    title_text="<b>Auto-Correlation Structure (USD/VND)</b>",
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(t=80, b=60, l=60, r=60),
)

fig.update_xaxes(title_text="Lag", gridcolor="#f2f2f2")
fig.update_yaxes(title_text="Correlation", gridcolor="#f2f2f2")

fig.show()


- **Mean Reversion (ACF):** A massive negative spike at **Lag 3** indicates an inverse 3-day memory. This is likely a "Step-Function" artifact of the 3-day intervention window in the VND pegged regime.
- **Predictable PACF:** Significant negative spike at **Lag 1** suggests a direct AR(1) component, providing the $(p,d,q)$ parameters for the ARIMA stage.
- **Volatility Continuity ($Sq-ACF$):** Positive, significant spikes (especially at **Lag 3**) prove that return magnitude is autocorrelated. Large swings today correlate with similar shocks 3 days later, formally justifying the Hybrid stage.

### **Core Time-Series Dependencies (USD/VND)**
- **ACF:** Detects Moving Average (MA) components and general periodicity
- **PACF:** Isolates direct Autoregressive (AR) dependencies, stripping intermediate effects
- **Squared ACF:** Confirms second-moment dependency (volatility clustering) for Hybrid justification


In [8]:
from statsmodels.tsa.stattools import acf, pacf

# 1. Prepare 3-way diagnostics
series = df_train["USDVND_RET"].dropna()
lags_to_show = 40

acf_vals = acf(series, nlags=lags_to_show)
pacf_vals = pacf(series, nlags=lags_to_show)
acf_sq_vals = acf(series**2, nlags=lags_to_show)
lags = np.arange(len(acf_vals))

# 2. Setup Subplots
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "<b>ACF (MA order)</b>",
        "<b>PACF (AR order)</b>",
        "<b>Sq-ACF (Volatility)</b>",
    ),
    horizontal_spacing=0.08,
)

# Plots
fig.add_trace(go.Bar(x=lags, y=acf_vals, marker_color="#2c3e50"), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=pacf_vals, marker_color="#34495e"), row=1, col=2)
fig.add_trace(go.Bar(x=lags, y=acf_sq_vals, marker_color="#d35400"), row=1, col=3)

# Significance Threshold (95% CI)
ci = 1.96 / np.sqrt(len(series))
for col in [1, 2, 3]:
    fig.add_hline(
        y=ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )
    fig.add_hline(
        y=-ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )

# 3. Styling
fig.update_layout(
    template="plotly_white",
    height=400,
    width=1250,
    showlegend=False,
    title_text="<b>USD/VND Temporal Dependency Structure</b>",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(t=80, b=50, l=50, r=50),
)
fig.update_xaxes(title_text="Lag")
fig.update_yaxes(
    range=[-1, 1], gridcolor="#f2f2f2"
)  # Standardize scale to see magnitude

fig.show()


- **Mean Reversion (ACF):** A massive negative spike at **Lag 3** indicates an inverse 3-day memory. This is likely a "Step-Function" artifact of the 3-day intervention window in the VND pegged regime.
- **Predictable PACF:** Significant negative spike at **Lag 1** suggests a direct AR(1) component, providing the $(p,d,q)$ parameters for the ARIMA stage.
- **Volatility Continuity ($Sq-ACF$):** Positive, significant spikes (especially at **Lag 3**) prove that return magnitude is autocorrelated. Large swings today correlate with similar shocks 3 days later, formally justifying the Hybrid stage.


**Cross-Comparison**
*   **Consistency:** The **Kurtosis (37.8)** from the table is perfectly visualized by the **Distribution spike** and the **Q-Q tail divergence**.
*   **Linearity vs. Volatility:** While the **Log-Returns (ACF)** oscillate (mean-reversion), the **Squared Returns (Sq-ACF)** are strictly positive, proving that **direction** is hard to predict but **intensity** (volatility) is highly persistent.
*   **The Lag 3 Signature:** The dominance of Lag 3 across ACF, PACF, and Sq-ACF points to a structural "3-day heartbeat" in the USD/VND data, which the Hybrid model is specifically designed to exploit.